# Hisaab Aunty: Hinglish training (LoRA)

Teaches your model to turn one message into ONE JSON intent. It does **not** teach tax or banking facts.

**Before you start**
1. Runtime menu: **Change runtime type**, pick **T4 GPU**, Save.
2. In Google Drive make a folder `Hisaab` with two things inside: your model folder `Sent_model` (config.json, tokenizer files, model.safetensors) and the folder `data` (train.jsonl, val.jsonl, test.jsonl from `prototype/training/data` on your PC).
3. Press **Runtime > Run all**. Allow Drive access when asked.

This notebook has not been run by the author (no GPU on their laptop). If a step fails, copy the last lines of the error to Claude.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Hisaab/Sent_model'            # your model folder
DATA = '/content/drive/MyDrive/Hisaab/data'                  # train.jsonl, val.jsonl, test.jsonl
OUT  = '/content/drive/MyDrive/Hisaab/Sent_model_hinglish'   # the trained model is saved here

import os
for p in (BASE + '/config.json', BASE + '/model.safetensors', DATA + '/train.jsonl', DATA + '/val.jsonl'):
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + p)

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate
!git clone -q https://github.com/sohail807/hisaab_aunty /content/repo
!ls /content/repo/prototype/training

## Train (about the longest step; a free T4 may take under an hour, which is an estimate)

In [ ]:
# runs in its own process so the GPU memory is fully freed afterwards
!python /content/repo/prototype/training/train_lora.py --base "$BASE" --data "$DATA" --out "$OUT" --epochs 2

## Quick check: does the new model answer in the right format?

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(OUT)
model = AutoModelForCausalLM.from_pretrained(OUT, torch_dtype=torch.float16, device_map='auto')
rows = [json.loads(l) for l in open(DATA + '/test.jsonl')][:12]
ok = 0
for r in rows:
    prompt = tok.apply_chat_template(r['messages'][:2], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=120, do_sample=False)
    got = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    gold = r['messages'][2]['content']
    same = False
    try: same = json.loads(got).get('intent') == json.loads(gold).get('intent')
    except Exception: pass
    ok += same
    print(('OK  ' if same else 'MISS'), r['messages'][1]['content'], '=>', got)
print(ok, 'of', len(rows), 'right intent on this small sample')

## Done
The trained model is in your Drive folder `Hisaab/Sent_model_hinglish` (about 6 GB). Download it to your PC, then:
1. Convert it: `convert_hf_to_gguf.py <folder> --outfile model-hinglish-f16.gguf --outtype f16`
2. Add it to `prototype/models.json`, start a `llama-server` for it on a new port.
3. Score it: `node training/eval_model.js --model <id> --n 70` and compare with the baseline (59% intent, 2 wrong writes).